## scenario -1 
**-Managed Catalog**
**-Managed Schema**
**-Managed Table**

**Managed Catalog**

In [0]:
%sql
create  catalog man_cata

**Managed Schema/Database**

In [0]:
%sql
create schema man_cata.man_schema

**Managed Table**

In [0]:
%sql
create table man_cata.man_schema.man_table
(
  id INT,
  name STRING
)
using delta

## scenario - 2
**-External Catalog**
**-Managed Schema**
**-Managed Table**

In [0]:
%sql
create catalog ext_cata
managed location 'abfss://mycontainer@storagemoderndbashu.dfs.core.windows.net/external_catalog'

In [0]:
%sql
create schema ext_cata.man_schema

In [0]:
%sql
create table ext_cata.man_schema.man_table2
(
  id int,
  name string
)
using delta

## scenario - 3 
**-External Catalog**
**-External Schema**
**-Managed Table**

In [0]:
%sql
create schema ext_cata.ext_schema
managed location 'abfss://mycontainer@storagemoderndbashu.dfs.core.windows.net/ext_schema'

In [0]:
%sql
create table ext_cata.ext_schema.man_table3
(
  id int,
  name string
)
using delta

## scenario - 4
**-External Table**

In [0]:
%sql
create table man_cata.man_schema.ext_table
(
  id int,
  name string
)
using delta
location 'abfss://mycontainer@storagemoderndbashu.dfs.core.windows.net/ext_table/man_table3'

#### Insert Data

In [0]:
%sql
insert into man_cata.man_schema.ext_table
values
(1,'Ashutosh'),
(2,'Acharya'),
(3,'Rajesh')

num_affected_rows,num_inserted_rows
3,3


#### DROP Managed Table

In [0]:
%sql
drop table man_cata.man_schema.man_table

#### UNDROP Managed Table

In [0]:
%sql
undrop table man_cata.man_schema.man_table

# Querying file using SELECT

In [0]:
%sql
select * from man_cata.man_schema.ext_table

id,name
1,Ashutosh
2,Acharya
3,Rajesh


In [0]:
%sql
select * from delta.`abfss://mycontainer@storagemoderndbashu.dfs.core.windows.net/ext_table/man_table3` where id =1

id,name
1,Ashutosh


# Permanent Views

In [0]:
%sql
create view man_cata.man_schema.view1 as
select * from delta.`abfss://mycontainer@storagemoderndbashu.dfs.core.windows.net/ext_table/man_table3` where id =1

org.apache.spark.sql.AnalysisException: [TABLE_OR_VIEW_ALREADY_EXISTS] Cannot create table or view `man_cata`.`man_schema`.`view1` because it already exists.
Choose a different name, drop or replace the existing object, add the IF NOT EXISTS clause to tolerate pre-existing objects, or add the OR REFRESH clause to refresh the existing streaming table. SQLSTATE: 42P07
	at org.apache.spark.sql.errors.QueryCompilationErrors$.viewAlreadyExistsError(QueryCompilationErrors.scala:3358)
	at org.apache.spark.sql.execution.command.CreateViewCommand.run(views.scala:246)
	at org.apache.spark.sql.execution.command.ExecutedCommandExec.$anonfun$sideEffectResult$2(commands.scala:84)
	at org.apache.spark.sql.execution.SparkPlan.runCommandWithAetherOff(SparkPlan.scala:178)
	at org.apache.spark.sql.execution.SparkPlan.runCommandInAetherOrSpark(SparkPlan.scala:189)
	at org.apache.spark.sql.execution.command.ExecutedCommandExec.$anonfun$sideEffectResult$1(commands.scala:84)
	at com.databricks.spark.util.Fra

In [0]:
%sql
select * from man_cata.man_schema.view1

id,name
1,Ashutosh
2,Acharya
3,Rajesh


# Temporary Views (it will remove once cluster is went off)

In [0]:
%sql
create or replace temp view temp_view as
select * from delta.`abfss://mycontainer@storagemoderndbashu.dfs.core.windows.net/ext_table/man_table3` where id =1

In [0]:
%sql
select * from temp_view

id,name
1,Ashutosh


# Volumes

##### Volumes provide capabilities for accessing, storing, governing and organizing files. While tables provide governance over tabular datasets, volumes add governance over non-tabular dataset. You can use volumes to store and access files in any format, including structured semi structured and unstructured data.
##### Volumes are siblings to tables, views, and other objects organized under a schema in unity catalog. A volume can be managed or external

**Creating a directory for Volume**

In [0]:
dbutils.fs.mkdirs('abfss://mycontainer@storagemoderndbashu.dfs.core.windows.net/volumes')

True

**Creating Volume**

In [0]:
%sql
create external volume man_cata.man_schema.extVolume
location 'abfss://mycontainer@storagemoderndbashu.dfs.core.windows.net/volumes'

**copy file for volume**

In [0]:
dbutils.fs.cp('abfss://mycontainer@storagemoderndbashu.dfs.core.windows.net/source/Fact_Sales_1.csv', 'abfss://mycontainer@storagemoderndbashu.dfs.core.windows.net/volumes/Fact_Sales_1')

True

In [0]:
%sql
select * from csv.`/Volumes/man_cata/man_Schema/extvolume/Fact_Sales_1`

_c0,_c1,_c2,_c3,_c4,_c5,_c6,_c7,_c8,_c9
transaction_id,transactional_date,product_id,customer_id,payment,credit_card,loyalty_card,cost,quantity,price
1,2021-05-04 02:00:00,P0494,4,visa,4041593010498829,F,17.33,2,18.29
2,2021-05-04 03:04:00,P0221,5,visa,4041596151234556,F,0.59,1,1.49
3,2021-05-04 03:56:00,P0625,5,visa,4041594885335898,F,5.15,3,5.89
4,2021-05-04 05:20:00,P0431,8,mastercard,5108753677552345,F,10.67,2,11.59
5,2021-05-04 05:45:00,P0058,5,mastercard,5108752372298261,T,11.38,2,12.39
6,2021-05-04 06:58:00,P0385,6,americanexpress,374288563442549,F,13.22,1,14.69
7,2021-05-04 07:03:00,P0575,4,visa,4041598869758,F,2.81,1,3.99
8,2021-05-04 07:45:00,P0187,5,americanexpress,374283491107967,F,4.17,1,4.89
9,2021-05-04 09:58:00,P0074,7,mastercard,5108753211196286,F,18.11,1,19.79
